<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-10c-realtime-service-slo-load-test-and-ci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 10 (continued) - Running a real-time model service: latency budget, load test, SLOs, CI and the on-call runbook

A model that scores well in a notebook is not yet a service. This lab builds a small HTTP prediction service with a feature lookup in front of the model, then does what a team does before putting it on a pager: **break the latency into a budget, load-test it at rising concurrency, state an SLO and its error budget, protect it with a timeout and a fallback, wire a CI workflow and a smoke test, and write the runbook.**

**Optional advanced-discussion lab** for Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Runs on the free Colab CPU in about two minutes. The service runs as a separate process started from the notebook.

*Original code written for this course. Practices follow the Google SRE Book and SRE Workbook (service level objectives, error budgets, burn-rate alerting) and the GitHub Actions documentation. The model is a stand-in: the Wisconsin breast-cancer table that ships with scikit-learn, used as a generic 30-feature scoring problem. What is measured here is the service around the model, not the model.*

In [ ]:
%pip install -q numpy scikit-learn requests pyyaml

In [ ]:
import time, json, threading, http.client, socket, subprocess, sys, os, tempfile
import numpy as np
np.random.seed(0)

SERVER_PY = r'''
import time, json, threading, sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutTimeout
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.3, random_state=0, stratify=data.target)
model = HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(Xtr, ytr)
ACC = float(model.score(Xte, yte)); MODEL_VERSION = "risk-gbm-2026-10-07"
FEATURE_STORE = {f"id{i}": row for i, row in enumerate(data.data)}      # stand-in for an online feature store (Redis, DynamoDB, ...)

class Cfg:
    store_delay_s = 0.001; slow_tail_p = 0.0; slow_tail_s = 0.0; timeout_s = None; fallback_score = 0.5
STAGES = {"fetch": [], "predict": [], "total": []}
lock = threading.Lock(); pool = ThreadPoolExecutor(max_workers=64); rng = np.random.default_rng(0)

def fetch_features(eid):
    d = Cfg.store_delay_s
    if Cfg.slow_tail_p and rng.random() < Cfg.slow_tail_p: d += Cfg.slow_tail_s
    time.sleep(d)
    return FEATURE_STORE[eid]

class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def log_message(self, *a): pass
    def send_obj(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code); self.send_header("Content-Type", "application/json"); self.send_header("Content-Length", str(len(body))); self.end_headers(); self.wfile.write(body)
    def do_GET(self):
        if self.path == "/health": self.send_obj(200, {"status": "ok", "model_version": MODEL_VERSION, "heldout_accuracy": round(ACC, 3)})
        elif self.path == "/metrics":                      # per-stage timings, then reset
            with lock: out = {k: list(v) for k, v in STAGES.items()}; [v.clear() for v in STAGES.values()]
            self.send_obj(200, out)
        else: self.send_obj(404, {"error": "not found"})
    def do_POST(self):
        n = int(self.headers.get("Content-Length", 0)); raw = self.rfile.read(n)
        if self.path == "/admin/config":                    # change a knob between experiments
            for k, v in json.loads(raw).items(): setattr(Cfg, k, v)
            return self.send_obj(200, {"ok": True})
        t0 = time.perf_counter()
        try:
            eid = json.loads(raw)["entity_id"]
        except Exception:
            return self.send_obj(422, {"error": "bad request", "model_version": MODEL_VERSION})
        if eid not in FEATURE_STORE: return self.send_obj(404, {"error": "unknown entity", "model_version": MODEL_VERSION})
        degraded = False; t1 = time.perf_counter()
        try:
            feats = pool.submit(fetch_features, eid).result(timeout=Cfg.timeout_s)
            t2 = time.perf_counter()
            score = float(model.predict_proba(feats.reshape(1, -1))[0, 1])
        except FutTimeout:
            t2 = time.perf_counter(); score, degraded = Cfg.fallback_score, True
        t3 = time.perf_counter()
        self.send_obj(200, {"score": round(score, 4), "degraded": degraded, "model_version": MODEL_VERSION})
        t4 = time.perf_counter()
        with lock: STAGES["fetch"].append(t2 - t1); STAGES["predict"].append(t3 - t2); STAGES["total"].append(t4 - t0)

srv = ThreadingHTTPServer(("127.0.0.1", int(sys.argv[1])), Handler); srv.daemon_threads = True
srv.serve_forever()
'''
path = os.path.join(tempfile.gettempdir(), "risk_service.py"); open(path, "w").write(SERVER_PY)
with socket.socket() as s_: s_.bind(("127.0.0.1", 0)); PORT = s_.getsockname()[1]
proc = subprocess.Popen([sys.executable, path, str(PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def get(p):
    c = http.client.HTTPConnection("127.0.0.1", PORT, timeout=10); c.request("GET", p); return json.loads(c.getresponse().read())
def post(p, obj):
    c = http.client.HTTPConnection("127.0.0.1", PORT, timeout=10); c.request("POST", p, json.dumps(obj), {"Content-Type": "application/json"}); r = c.getresponse(); return r.status, json.loads(r.read())

for _ in range(100):                                    # wait for the service to come up (it trains its model first)
    try: print("health:", get("/health")); break
    except Exception: time.sleep(0.3)
print("predict:", post("/predict", {"entity_id": "id7"}))
print("bad body ->", post("/predict", {"wrong": 1})[0], "| unknown entity ->", post("/predict", {"entity_id": "nope"})[0])

## Part 2 - Where does the time go? A latency budget

Before optimising anything, split one request into stages. Here: the feature fetch, the model call, and everything else (parsing, serialising, the HTTP layer). The service runs in its **own process** (like a real one) and exposes the per-stage timings at `/metrics`. Percentiles, not averages: a user who sees the slow request does not care that the mean was fine.

In [ ]:
def pct(a, q): return float(np.percentile(a, q)) * 1000

def call(conn, eid):
    t0 = time.perf_counter()
    conn.request("POST", "/predict", json.dumps({"entity_id": eid}), {"Content-Type": "application/json"})
    r = conn.getresponse(); body = r.read(); return time.perf_counter() - t0, r.status, json.loads(body)

get("/metrics")                                           # reset
conn = http.client.HTTPConnection("127.0.0.1", PORT)
for i in range(300): call(conn, f"id{i % 500}")
ST = get("/metrics")
print(f"{'stage':10}{'p50 ms':>9}{'p95 ms':>9}{'p99 ms':>9}")
for k in ("fetch", "predict", "total"):
    print(f"{k:10}{pct(ST[k], 50):>9.2f}{pct(ST[k], 95):>9.2f}{pct(ST[k], 99):>9.2f}")
other = np.array(ST["total"]) - np.array(ST["fetch"]) - np.array(ST["predict"])
print(f"{'other':10}{pct(other, 50):>9.2f}{pct(other, 95):>9.2f}{pct(other, 99):>9.2f}   (parse, serialise, HTTP write)")

**Read it.** For a small tree model the model call is only a slice of the request; the feature lookup (a network round trip in real life) and the web layer are the rest. Budget each stage separately and attack the biggest one first: a faster model cannot repair a slow store.

## Part 3 - Load test: find the knee

A closed-loop load test runs *N* clients that each send a request, wait for the answer, and send the next. Raising *N* shows how throughput and tail latency respond. We report errors too: a service that answers fast with 500s is not fast.

In [ ]:
def load_test(concurrency, seconds=3.0, store_tail=False):
    get("/metrics")                                       # reset the service's stage timers
    lat, errs, degraded = [], [0], [0]; lk = threading.Lock(); stop = time.time() + seconds
    def client(seed):
        conn = http.client.HTTPConnection("127.0.0.1", PORT, timeout=10); i = seed
        while time.time() < stop:
            try:
                dt, st, body = call(conn, f"id{i % 500}"); i += 7
                with lk:
                    lat.append(dt)
                    if st != 200: errs[0] += 1
                    if body.get("degraded"): degraded[0] += 1
            except Exception:
                with lk: errs[0] += 1
                conn = http.client.HTTPConnection("127.0.0.1", PORT, timeout=10)
    ths = [threading.Thread(target=client, args=(s,)) for s in range(concurrency)]
    t0 = time.time(); [t.start() for t in ths]; [t.join() for t in ths]; wall = time.time() - t0
    return {"conc": concurrency, "rps": len(lat) / wall, "p50": pct(lat, 50), "p95": pct(lat, 95), "p99": pct(lat, 99),
            "err%": 100 * errs[0] / max(len(lat) + errs[0], 1), "over150%": 100 * float(np.mean(np.array(lat) * 1000 > 150)), "degraded%": 100 * degraded[0] / max(len(lat), 1), "n": len(lat)}

print(f"{'clients':>8}{'req/s':>9}{'p50 ms':>9}{'p95 ms':>9}{'p99 ms':>9}{'errors %':>10}")
base_rows = []
for n in (1, 4, 16, 64):
    r = load_test(n); base_rows.append(r)
    print(f"{r['conc']:>8}{r['rps']:>9.0f}{r['p50']:>9.1f}{r['p95']:>9.1f}{r['p99']:>9.1f}{r['err%']:>10.2f}")

**Read it.** One client already gets the service's best throughput; adding clients does not raise it, it only lengthens the queue: median latency climbs from about 3 ms to 15, 53 and 141 ms, and at 64 clients p99 reaches seconds and requests start to fail. The service is a single Python process, and its Python code runs one thread at a time (the global interpreter lock), so it can do only so much per second. This is the **knee**: past it, extra load buys latency, not work. **Little's law** ties the numbers together: clients in flight = throughput x latency (at 16 clients, about 276 req/s x 0.053 s = 14.6, close to 16). To serve more, add worker *processes* or replicas behind a load balancer, then repeat this test; never promise a latency without saying at what load it was measured. (Client and load generator share this machine, so read the shape rather than the absolute numbers; a real test uses a separate generator such as Locust, k6 or wrk.)

## Part 4 - A slow dependency, and how a timeout plus a fallback bounds the damage

Now make 2% of feature lookups slow (200 ms), as happens when a cache node is rebuilding. Run the same load three ways: healthy, slow tail with **no timeout**, and slow tail with a **30 ms timeout and a fallback score** (the response is marked `degraded` so downstream code and dashboards can see it).

In [ ]:
rows = {}
post("/admin/config", {"slow_tail_p": 0.0, "timeout_s": None}); rows["healthy"] = load_test(16)
post("/admin/config", {"slow_tail_p": 0.02, "slow_tail_s": 0.200, "timeout_s": None}); rows["2% slow lookups, no timeout"] = load_test(16)
post("/admin/config", {"timeout_s": 0.030}); rows["2% slow lookups, 30 ms timeout + fallback"] = load_test(16)
post("/admin/config", {"slow_tail_p": 0.0, "timeout_s": None})
print(f"{'scenario (16 clients)':46}{'req/s':>8}{'p50':>7}{'p95':>7}{'p99':>8}{'degraded %':>12}")
for k, r in rows.items():
    print(f"{k:46}{r['rps']:>8.0f}{r['p50']:>7.1f}{r['p95']:>7.1f}{r['p99']:>8.1f}{r['degraded%']:>12.2f}")

**Read it.** Without a timeout the slow tail goes straight into p99 (and, because workers are held, can queue other requests behind it). With a timeout the worst case is capped near the timeout, at the price of answering about 2% of requests with a default score instead of a real one. That is a *product* decision (is a default score acceptable for this use?), so it belongs in the design document, not only in the code. Always count and alert on the degraded share.

## Part 5 - SLOs and the error budget

An **SLO** is a promise stated as a measurable target over a window, for example: *99% of `/predict` requests complete in under 150 ms, and 99.9% of requests succeed, over 28 days.* The **error budget** is what the promise leaves you: 0.1% of the window. When the budget is being spent too fast, you alert; when it is gone, you stop shipping features and fix reliability.

In [ ]:
def budget_minutes(slo, days=28): return (1 - slo) * days * 24 * 60
def burn_rate(error_ratio, slo): return error_ratio / (1 - slo)          # 1.0 = spending the budget exactly as fast as it accrues

for slo in (0.99, 0.999, 0.9999):
    print(f"SLO {slo:.2%}: error budget over 28 days = {budget_minutes(slo):7.1f} minutes of full outage")

print("\nBurn-rate alert thresholds for a 99.9% / 30-day SLO (Google SRE Workbook, multiwindow approach):")
for rate, window_h in ((14.4, 1), (6, 6), (1, 72)):
    spent = rate * window_h / (30 * 24)
    print(f"  burn rate {rate:>4}x over {window_h:>2} h  -> {spent:.1%} of the 30-day budget gone in that window")

print("\nLatency SLO check (target: 99% of requests under 150 ms), measured in Part 4:")
for name, r in rows.items():
    ok = "meets" if r["over150%"] <= 1.0 else "MISSES"
    print(f"  {name:46} {r['over150%']:>5.2f}% of requests over 150 ms -> {ok} the target")

**Why burn rate, not a raw error count.** A burn rate of 1x means "at this pace the budget lasts exactly the window". A rate of 14.4x sustained for one hour spends 2% of a 30-day budget, which is worth waking someone; a slow 1x simmer is a ticket for working hours. The pair of thresholds keeps the pager quiet for blips and loud for real incidents - which is also what makes an on-call rotation bearable.

## Part 6 - CI/CD and a smoke test

A change to a model service should pass the same gate every time before it is built into an image. Below, the workflow file (GitHub Actions) and a shell smoke test. The workflow is parsed and checked for its required structure here; whether it runs depends on your repository, so treat it as a starting point.

In [ ]:
import textwrap, yaml, os, subprocess, shutil, tempfile
WORKFLOW = textwrap.dedent('''
name: model-service-ci
on:
  pull_request:
  push:
    branches: [main]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip
      - run: pip install -r requirements.txt
      - name: Unit tests (fast)
        run: pytest -q tests/unit
      - name: Start the service and smoke-test it
        run: bash scripts/smoke_test.sh
      - name: Gate - held-out accuracy must not regress
        run: python scripts/check_metrics.py --min-accuracy 0.93
  build:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: docker build -t model-service:${{ github.sha }} .
''')
wf = yaml.safe_load(WORKFLOW)
trig = wf.get("on", wf.get(True))                     # YAML 1.1 reads the bare key `on` as the boolean True
print("workflow parsed. triggers:", list(trig), "| jobs:", list(wf["jobs"]), "| build waits for:", wf["jobs"]["build"]["needs"])
assert "pull_request" in trig and wf["jobs"]["build"]["needs"] == "test"
print("steps in test job:", [s.get("name", s.get("uses", s.get("run"))) for s in wf["jobs"]["test"]["steps"]])

In [ ]:
SMOKE = '''#!/usr/bin/env bash
set -euo pipefail
URL="${1:-http://127.0.0.1:%PORT%}"
curl -fsS "$URL/health" | grep -q '"status": "ok"'
OUT=$(curl -fsS -X POST "$URL/predict" -H 'Content-Type: application/json' -d '{"entity_id": "id7"}')
echo "$OUT" | grep -q '"model_version"'
CODE=$(curl -s -o /dev/null -w '%{http_code}' -X POST "$URL/predict" -H 'Content-Type: application/json' -d 'not json')
test "$CODE" = "422"
echo "smoke test passed"
'''.replace("%PORT%", str(PORT))
path = os.path.join(tempfile.gettempdir(), "smoke_test.sh"); open(path, "w", newline="\n").write(SMOKE)
if shutil.which("bash") and shutil.which("curl"):
    r = subprocess.run(["bash", "smoke_test.sh"], cwd=tempfile.gettempdir(), capture_output=True, text=True, timeout=60)
    print("exit code", r.returncode, "|", (r.stdout + r.stderr).strip()[-200:])
    if r.returncode: print("(On Windows, `bash` may be WSL, whose 127.0.0.1 is a different machine from the notebook's. Colab and Linux are unaffected; on Windows use Git Bash.)")
else:
    print("bash or curl not found here; the script is saved at", path, "- run it where both exist (Colab has both).")

## Part 7 - The on-call runbook (write it before the first page)

A runbook turns a 3 a.m. alert into a checklist. Keep it short enough to be used half asleep.

| Step | Question | First actions |
|---|---|---|
| 1. Acknowledge | Which alert fired, since when? | Note the time; open the dashboard linked from the alert. |
| 2. Scope | One endpoint or all? One model version or all? | Compare error rate and p99 by `model_version` and by status code. |
| 3. Recent change? | Was anything deployed or retrained in the last hour? | If yes, **roll back first, investigate second** (route traffic to the previous model version). |
| 4. Dependency? | Is the feature store / upstream data slow or down? | Check the degraded share; confirm timeouts and fallbacks are working; shed load if queues grow. |
| 5. Communicate | Who is affected? | Post status; set the next update time. |
| 6. Close | Is the SLO burn back to normal? | Write a short postmortem: timeline, cause, what detected it, what would have detected it sooner. |
| 7. Prevent | What stops this class of failure? | Add a test or a monitor for it (as in the earlier discussion, every incident becomes a new test). |

**Healthy on-call practice:** a primary and a secondary, schedules published well ahead, alerts only for conditions that need a human *now* (page on SLO burn, ticket for the rest), and a postmortem culture that blames the system rather than the person.

In [ ]:
proc.terminate(); proc.wait(timeout=10); print("service stopped")

## What to hand in

Your latency-budget table, the load-test table with the knee marked, the three-scenario slow-dependency table with a one-paragraph decision on whether a degraded default score is acceptable for your use, your SLO statement with its error budget in minutes, and a runbook adapted to your own service.